# 10年定着予測 - TabPFN側の増強（65_）

## 位置づけ: 新最良 0.508793 の上積み

第97節で **プールC × TabPFN(top150) が Public 0.508793** で新最良を更新した
（`50_` 0.513108比 **-0.004315**、同一レシピの実行間ばらつき0.0023の約2倍＝本物の改善）。

ただし**TabPFN側はまだ3シードしか使っていない**。AutoGluon側は8実行平均で既にそれなりに
平均化されている一方、TabPFN側には分散が残っている。ここを厚くするのが最も費用対効果が高い。

## やること

1. **シードを増やす**（既定5、`N_SEEDS`で調整可）
2. **特徴量数 K を複数使って平均する**（既定 100 / 150 / 200）

`63_`で top150(0.5076) が full441(0.5221) より明確に良いと分かったが、
**100/150/200 のどれが最適かはノイズの範囲**である可能性が高い。
そこで**どれか1つを検証スコアで選ぶのではなく、3つとも使って平均する**。
これは[[ablation-cannot-settle-feature-blocks]]の「勝者の呪い」を回避する設計であり、
同時に特徴量サブセットの違いによる多様性も得られる。

3. AutoGluonプール（保存済み8実行平均）と **w_AG=0.70** でブレンドして提出ファイルを作る

## ⚠️ GPUが使えない前提での設計

TabPFNはGPU推奨だがCPUでも動く（`ignore_pretraining_limits=True`で件数制限も回避）。
CPUだと1フィットあたり数分かかりうるため:

- **1フィットごとに所要時間を表示**する。想定より遅ければ`N_SEEDS`を減らして再実行すればよい
- **1フィットごとに予測を即保存**する。途中で中断しても、それまでの成果は失われず、
  最後のセルは「保存済みのものだけ」を拾って平均する
- 検証(535名)より**全件学習→Test予測の方が重要**なので、そちらを先に回す

## 重みについて（重要）

**w_AG=0.70 は変えない。** `63_`の走査で決めた値であり、
**Publicで重みを走査するのはPrivate評価に対する過学習**（勝者の呪い）になる。
[[private-lb-variance-strategy]]の通り、本コンペはPrivateで最終順位が決まる。


In [ ]:
# ⚠️ tabpfn はバージョン固定が必須。素の `pip install tabpfn` は最新(8.x)を入れ、
#    Prior Labs のライセンス同意＋APIトークンを要求して TabPFNLicenseError で落ちる。
#    2.x 系が TabPFN v2 世代で Apache 2.0ベース・ライセンスゲート無し。
!pip install -q catboost optuna "tabpfn==2.2.1"

# 出る「ERROR: pip's dependency resolver...」は依存解決の警告であって失敗ではない。
# huggingface-hub 0.36.x への降格は tabpfn 2.2.1 の要求(huggingface-hub<1)通りで正しい。
#
# ⚠️ 既に別バージョンが入っている場合は、このセルの後で
#    「ランタイム → セッションを再起動」してから最初のセルに戻ること。


In [ ]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

In [ ]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


In [ ]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [ ]:
SCRIPT_NAME = "65_tabpfn_scaleup"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


In [ ]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")


## 0. 早期退職者の特定

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、Test には0名（[[test-set-is-survivor-filtered]]）。


In [ ]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れているので調査すること"


## 1. 基本特徴量関数の定義（split非依存、`51_`と同一ロジック）

In [ ]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")


In [ ]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")


## 2. テキストTF-IDF（A_v1、`51_`と同一・継続採用）

In [ ]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")


## 3. 四半期/加速度特徴量（D_expanded、`51_`と同一・継続採用）

In [ ]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")


## 4. Persona単位の基本特徴量（`51_`と同一）

In [ ]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、v1=`27_`のPublic確認済み版 / v2=抽出拡張版、`51_`と同一）

`51_`と同じくv1/v2両方を生成するが、実際に特徴量として使うのはv2（`BLOCK={"L2"}`）のみ
（`28_`以降ずっとv2が現在の最良）。


In [ ]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 50_: 見出しがない書式B（276件、5.24%）のフォールバック（49_で確認済み・Public -0.0022〜-0.0035）。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")


## 6. 部署Target Encoding（リーク対策済）と `prepare_split` 関数（`51_`と同一）

In [ ]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる（51_と完全に同一ロジック）。'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L1" in extra_blocks:
        tf = tf.merge(train_reloc_v1, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v1, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了")


## 7. 特徴量の組み立て（`51_`と同一、`BLOCK={"L2"}`固定）

In [ ]:
def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


BLOCK = {"L2"}   # 28_のL_v2_extended（現在の最良）に固定

logger.info("=" * 60)
logger.info("[検証用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[提出用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"main_train={len(ag_train_80b)}, main_valid(生存者)={len(ag_val_surv)}")
logger.info(f"全件={len(ag_full)}")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80b))}")
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"


## 8. 特徴量グループの棚卸し（`51_`から移植、内容は同一）

In [ ]:
ALL_FEATS = set(_feature_cols(ag_train_80b))

DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "derived":   DERIVED_COLS,
}

FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


## 9. 設定（CPU前提。遅ければ N_SEEDS を減らす）

In [ ]:
import time, warnings
import torch, tabpfn
from tabpfn import TabPFNClassifier
from importlib.metadata import version as _pkgver
warnings.filterwarnings("ignore")

_disk, _live = _pkgver("tabpfn"), tabpfn.__version__
if not _live.startswith("2."):
    if _disk.startswith("2."):
        raise RuntimeError(
            f"ディスク上は tabpfn=={_disk}（正しい）だが、カーネルには {_live} が残っています。\n"
            "→ ランタイム → セッションを再起動 してから最初のセルに戻ってください。")
    raise RuntimeError(f"tabpfn=={_disk}。v2系(2.x)が必要。セル1を実行して再起動してください。")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"tabpfn {_live} / device = {DEVICE}")
if DEVICE == "cpu":
    print("ℹ️ CPU実行。1フィットに数分かかる場合があります。")
    print("   各フィットの所要時間を表示するので、遅すぎる場合は N_SEEDS を減らして再実行してください。")
    print("   予測はフィットごとに保存されるので、途中で止めても成果は残ります。")

# --- ここを調整する ---
N_SEEDS = 5                      # TabPFNのシード数（63_は3。CPUが遅ければ 3 に戻す）
FEATURE_COUNTS = [100, 150, 200] # 重要度上位K列。どれか1つを選ばず全部使って平均する
W_AG = 0.70                      # AutoGluonプール側の重み。63_で決めた値。変えない

ALL_SEEDS = [42, 2024, 7, 1234, 99, 555, 31337, 2718, 123, 999]
SEEDS = ALL_SEEDS[:N_SEEDS]
print(f"\nTabPFN: {len(SEEDS)}シード × {len(FEATURE_COUNTS)}構成 = {len(SEEDS)*len(FEATURE_COUNTS)}フィット")
print(f"シード: {SEEDS} / 特徴量数: {FEATURE_COUNTS} / ブレンド重み w_AG={W_AG}")

A_PARAMS = {"depth": 4, "learning_rate": 0.03518359458951149, "l2_leaf_reg": 2.217690447016724,
            "border_count": 218, "bagging_temperature": 0.6787467566574921,
            "random_strength": 1.438494697238285}
ITER_FIXED = 560
FEATS_ALL = _feature_cols(ag_train_80b)
assert len(FEATS_ALL) == 441, f"{len(FEATS_ALL)}列（441列のはず）"


## 10. CatBoost で重要度を出す（top-K 特徴量セットの作成用）

`63_`と同じ手順。検証スコアの比較基準としても使う。


In [ ]:
def cb_fit(train_df, feats, predict_dfs, seeds, n_iter=ITER_FIXED):
    obj = [c for c in feats if train_df[c].dtype == "object"]
    Xtr, ytr = train_df[feats].fillna(-999), train_df[TARGET_COL]
    outs = [[] for _ in predict_dfs]; models = []
    for s in seeds:
        m = cb.CatBoostClassifier(**A_PARAMS, iterations=n_iter, random_seed=s, verbose=False,
                                  cat_features=obj, task_type="CPU")
        m.fit(Xtr, ytr); models.append(m)
        for k, df in enumerate(predict_dfs):
            outs[k].append(m.predict_proba(df[feats].fillna(-999))[:, 1])
    return [np.mean(o, axis=0) for o in outs], models


y_val = ag_val_surv[TARGET_COL].values
t0 = time.time()
(cb_val,), cb_models = cb_fit(ag_train_80b, FEATS_ALL, [ag_val_surv], [42, 2024, 7])
CB_VAL = log_loss(y_val, cb_val)
print(f"CatBoost baseline val = {CB_VAL:.6f}（{time.time()-t0:.0f}秒）")

imp = pd.Series(np.mean([m.get_feature_importance() for m in cb_models], axis=0),
                index=FEATS_ALL).sort_values(ascending=False)
imp.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_catboost_importance.csv")
FEATURE_SETS = {f"top{k}": imp.head(k).index.tolist() for k in FEATURE_COUNTS}
for k, v in FEATURE_SETS.items():
    print(f"  {k}: {len(v)}列")


## 11. TabPFN の学習（全件学習 → Test予測を先に回す）

**検証より全件学習→Test予測の方が重要**なので先に回す。
1フィットごとに `.npy` に保存するので、途中で中断してもそこまでの成果は残る。


In [ ]:
def to_matrix(train_df, other_dfs, feats):
    """TabPFN用。NaNは埋めない（ネイティブに扱える）。カテゴリは序数コード化。"""
    obj = [c for c in feats if train_df[c].dtype == "object"]
    frames = [train_df] + list(other_dfs)
    cmap = {c: {v: i for i, v in enumerate(sorted(pd.concat([f[c].astype(str) for f in frames]).unique()))}
            for c in obj}
    out = []
    for f in frames:
        M = f[feats].copy()
        for c in obj:
            M[c] = f[c].astype(str).map(cmap[c]).astype(float)
        out.append(M.astype(np.float32).values)
    return out, [feats.index(c) for c in obj]


def tabpfn_one(train_df, other_dfs, feats, seed):
    (Mtr, *Mo), cat_idx = to_matrix(train_df, other_dfs, feats)
    for extra in ({"categorical_features_indices": cat_idx, "ignore_pretraining_limits": True},
                  {"categorical_features_indices": cat_idx}, {}):
        try:
            clf = TabPFNClassifier(device=DEVICE, random_state=seed, **extra); break
        except TypeError:
            continue
    clf.fit(Mtr, train_df[TARGET_COL].values)
    return [clf.predict_proba(M)[:, 1] for M in Mo]


PRED_DIR = OUTPUT_DIR
def _path(kind, name, seed):
    return PRED_DIR / f"{TODAY}_{SCRIPT_NAME}_{kind}_{name}_seed{seed}.npy"

print("=== 全件学習 → Test予測 ===")
done = 0; total = len(FEATURE_SETS) * len(SEEDS)
for name, feats in FEATURE_SETS.items():
    for seed in SEEDS:
        p = _path("test", name, seed)
        if p.exists():
            print(f"  [{name} seed{seed}] 保存済みをスキップ"); done += 1; continue
        t0 = time.time()
        (pr,) = tabpfn_one(ag_full, [test_features_full], feats, seed)
        np.save(p, pr); done += 1
        el = time.time() - t0
        print(f"  [{name} seed{seed}] {el:6.1f}秒  予測平均={pr.mean():.4f}  ({done}/{total})"
              f"  残り推定 {el*(total-done)/60:.0f}分")
        logger.info(f"[{name} seed{seed}] test予測を保存")
print("全件学習→Test予測 完了")


## 12. 検証（参考情報。採否には使わない）

[[validation-asymmetry]]の通り検証の「改善」は根拠にならないので、
ここは**TabPFNが壊れていないかの健全性確認**として見る。時間がなければスキップしてよい。


In [ ]:
RUN_VALIDATION = True   # 時間がなければ False にしてスキップしてよい

if RUN_VALIDATION:
    print("=== 検証(生存者535名) ===")
    val_acc = {}
    for name, feats in FEATURE_SETS.items():
        ps = []
        for seed in SEEDS:
            p = _path("val", name, seed)
            if p.exists():
                ps.append(np.load(p)); continue
            t0 = time.time()
            (pv,) = tabpfn_one(ag_train_80b, [ag_val_surv], feats, seed)
            np.save(p, pv); ps.append(pv)
            print(f"  [{name} seed{seed}] {time.time()-t0:6.1f}秒")
        v = np.mean(ps, axis=0); val_acc[name] = v
        print(f"  {name}: TabPFN単体 val={log_loss(y_val, v):.6f} "
              f"(CatBoost {CB_VAL:.6f} 比 {log_loss(y_val, v)-CB_VAL:+.6f})")
    V = np.mean(list(val_acc.values()), axis=0)
    print(f"\n  {len(FEATURE_SETS)}構成の平均: val={log_loss(y_val, V):.6f}")
    for w in [0.5, 0.6, 0.7, 0.8, 1.0]:
        print(f"    w_CatBoost={w:.1f}: {log_loss(y_val, np.clip(w*cb_val+(1-w)*V,1e-9,1-1e-9)):.6f}")
    print("  ※ ここでのCatBoostは単層441列。実際のブレンド相手はAutoGluonプールなので参考値。")


## 13. TabPFN予測の集約 と AutoGluonプールとのブレンド

In [ ]:
import glob
found = {}
for name in FEATURE_SETS:
    for seed in SEEDS:
        p = _path("test", name, seed)
        if p.exists():
            found[f"{name}_s{seed}"] = np.load(p)
assert found, "TabPFNのTest予測が1つも無い"
TP = np.mean(list(found.values()), axis=0)
print(f"TabPFN: {len(found)}フィットを平均  予測平均={TP.mean():.4f}")
print(f"  内訳: {sorted(found)}")

# --- AutoGluonプール（保存済み実行の平均）---
OUT_ROOT = PROJECT_ROOT / "data" / "output"
def _csv(pat):
    fs = sorted(OUT_ROOT.glob(pat))
    return (pd.read_csv(fs[-1], header=None, names=[ID_COL, "p"]).set_index(ID_COL)["p"]
            .reindex(test_features_full.index).values) if fs else None
def _npy(pat):
    fs = sorted(OUT_ROOT.glob(pat)); return np.load(fs[-1]) if fs else None

AG = {}
for nm, kind, pat in [
    ("50_", "csv", "*/*_50_autogluon_memofix_AG50_full441_weighted.csv"),
    ("51_", "csv", "*/*_51_autogluon_catboost_bias_AG51_full441_weighted.csv"),
    ("53_", "csv", "*/*_53_autogluon_dystack_AG53_full441_weighted.csv"),
    ("61_", "csv", "*/*_61_autogluon_extended_time_AG61_full441_weighted.csv"),
    ("62_bag16", "csv", "*/*_62_autogluon_seed_averaging_AG62_bag16_weighted.csv"),
    ("62_s42", "npy", "*/*_62_autogluon_seed_averaging_full441_seed42_weighted_testpreds.npy"),
    ("62_s2024", "npy", "*/*_62_autogluon_seed_averaging_full441_seed2024_weighted_testpreds.npy"),
    ("62_s7", "npy", "*/*_62_autogluon_seed_averaging_full441_seed7_weighted_testpreds.npy"),
]:
    v = _csv(pat) if kind == "csv" else _npy(pat)
    if v is not None and not np.isnan(v).any():
        AG[nm] = v
# 64_ を実行済みならその分も自動で取り込む
for f in sorted(OUT_ROOT.glob("*/*_64_autogluon_more_seeds_full441_seed*_weighted_testpreds.npy")):
    AG[f"64_{f.stem.split('seed')[-1].split('_')[0]}"] = np.load(f)
print(f"\nAutoGluonプール: {len(AG)}実行  {sorted(AG)}")
assert len(AG) >= 8, f"AutoGluonプールが{len(AG)}本しかない"
POOL = np.mean(list(AG.values()), axis=0)

print(f"\n相関(TabPFN, プール) = {np.corrcoef(TP, POOL)[0,1]:.4f}  MAD = {np.abs(TP-POOL).mean():.5f}")
blend = W_AG * POOL + (1 - W_AG) * TP
label = f"pool{len(AG)}_tabpfn{len(found)}_w{int(W_AG*100)}"
path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{label}.csv"
pd.DataFrame({ID_COL: test_features_full.index, TARGET_COL: blend}).to_csv(path, index=False, header=False)
print(f"\n提出ファイル: {path.name}（予測平均={blend.mean():.4f}）")

prev = _csv("*/20260816_pool_tabpfn_blend_w70.csv")
if prev is not None:
    print(f"  現最良(0.508793)との MAD={np.abs(blend-prev).mean():.5f} "
          f"相関={np.corrcoef(blend,prev)[0,1]:.5f}")
logger.info(f"完了: {label}")


## 14. 提出方針

- **第一候補は本セルが出力した `pool{N}_tabpfn{M}_w70`**。
  現最良（Public 0.508793）と同じレシピで、**TabPFN側のフィット数だけを増やした**もの
- **重み w_AG=0.70 は変えない。** `63_`の走査で決めた値であり、
  Publicで重みを走査するのはPrivate評価に対する過学習になる（[[private-lb-variance-strategy]]）
- 採否はPublicで見るが、**最終提出（Private）はこのブレンド系を選ぶ**。
  AutoGluonプール平均＋TabPFN平均という二重の平均化により分散が最小化されている

### 期待値について

現最良はTabPFN 3フィットで出した。本ノートブックは最大15フィット（5シード×3構成）に増やす。
第94節で「AutoGluon側は3本→8本で-0.0013改善」した実績があるので、
TabPFN側の増強も同程度の効きが期待できる。ただし**改善幅は小さいはず**なので、
Publicで悪化して見えてもノイズの範囲であり、分散が下がっている以上
Privateでは有利という判断は変わらない。
